In [1]:
# import libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [3]:
# load data
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df = pd.read_csv(url)

df_country = (
    df.drop(columns=["Province/State", "Lat", "Long"])
    .groupby("Country/Region")
    .sum()
)

series_honduras = (
    df_country.loc["Honduras"]
    .drop("Country/Region", errors="ignore")
    .astype(float)
)

first_case_idx = (series_honduras > 0).idxmax()
series_honduras = series_honduras.loc[first_case_idx:].reset_index(drop=True)

In [6]:
# create 5 day lag dataframe
df_honduras = pd.DataFrame({"y_t": series_honduras})
for lag in range(1, 6):
  df_honduras[f"y_lag_{lag}"] = series_honduras.shift(lag)

df_honduras = df_honduras.dropna().reset_index(drop=True)

features = ["y_lag_1", "y_lag_2", "y_lag_3", "y_lag_4", "y_lag_5"]
X_all = df_honduras[features].values
y_all = df_honduras["y_t"].values

In [7]:
# split in half
split_idx = len(df_honduras) // 2
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_test, y_test = X_all[split_idx:], y_all[split_idx:]
test_index = df_honduras.index[split_idx:]